# 02 - Exploratory analysis and stationarity

Time-series components, STL decomposition, ADF/KPSS tests and ACF/PACF (assignment Part 1).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from appliance_energy.config import *
from appliance_energy import data, eda, features, evaluation, plotting

In [ ]:
hourly = data.resample_hourly(data.load_raw())
y = hourly[TARGET]
eda.plot_series_overview(hourly, TARGET)
display(Image(str(FIGURE_DIR / 'fig01_eda_overview.png')))

The series has a clear **daily cycle** (low overnight, peaks around midday and 18:00) and a mild weekly effect (weekends higher), with no visible long-term trend.

In [ ]:
print(eda.stl_decomposition(y, period=24))
print(eda.stl_decomposition(y, period=168))
display(Image(str(FIGURE_DIR / 'fig02_stl_period24.png')))

Seasonal strength is ~0.32 (daily) and ~0.37 (weekly): seasonality is present but the remainder dominates - the series is noisy and spiky.

## Stationarity tests (train period only)

In [ ]:
train = y.iloc[:-TEST_STEPS]
tests = pd.DataFrame([
    eda.adf_test(train, 'raw'), eda.kpss_test(train, 'raw'),
    eda.adf_test(train.diff(), 'diff(1)'),
    eda.kpss_test(train.diff(), 'diff(1)'),
    eda.adf_test(train.diff(24), 'sdiff(24)'),
    eda.kpss_test(train.diff(24), 'sdiff(24)'),
])
tests

ADF rejects a unit root (p < 0.05) and KPSS does not reject stationarity: the hourly series is already **level-stationary**. Differencing is therefore not strictly required, although the AIC grid search later marginally prefers d = 1 (differences of near-stationary series remain stationary).

In [ ]:
eda.acf_pacf_plots(train, 'fig03_acf_pacf_raw.png', 'raw')
eda.differencing_figure(y)
display(Image(str(FIGURE_DIR / 'fig03_acf_pacf_raw.png')))

The ACF shows clear spikes at lags 24, 48, 72 - the daily seasonal component that motivates seasonal models.